# im2col Convolution and Raw-vs-Conv Comparison
Implement matrix-multiply convolution and compare handcrafted convolution features with raw pixels on the digits dataset.


In [1]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score


In [2]:
def load_split(seed=0):
    d = load_digits()
    X_img = d.images.astype(float)
    y = d.target
    return train_test_split(
        X_img, y, test_size=0.25, random_state=seed, stratify=y
    )


def im2col(img, k):
    img = np.asarray(img, dtype=float)
    H, W = img.shape
    oh, ow = H - k + 1, W - k + 1
    cols = np.zeros((oh * ow, k * k), dtype=float)

    row = 0
    for i in range(oh):
        for j in range(ow):
            cols[row] = img[i:i + k, j:j + k].reshape(-1)
            row += 1

    return cols


def conv_via_im2col(img, kernel):
    img = np.asarray(img, dtype=float)
    kernel = np.asarray(kernel, dtype=float)
    k = kernel.shape[0]
    oh = img.shape[0] - k + 1
    ow = img.shape[1] - k + 1
    values = im2col(img, k) @ kernel.reshape(-1)
    return values.reshape(oh, ow)


def relu(x):
    return np.maximum(0, np.asarray(x, dtype=float))


def max_pool2d(fmap, size=2):
    fmap = np.asarray(fmap, dtype=float)
    H, W = fmap.shape
    out = np.zeros((H // size, W // size), dtype=float)

    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            block = fmap[i*size:(i+1)*size, j*size:(j+1)*size]
            out[i, j] = np.max(block)

    return out


def conv_features(X_img, kernels, pool=2):
    feature_rows = []

    for img in X_img:
        image_parts = []
        for kernel in kernels:
            conv_map = conv_via_im2col(img, kernel)
            activated = relu(conv_map)
            pooled = max_pool2d(activated, size=pool)
            image_parts.append(pooled.reshape(-1))
        feature_rows.append(np.concatenate(image_parts))

    return np.asarray(feature_rows, dtype=float)


def default_kernels():
    sobel_x = np.array([[1, 0, -1],
                        [2, 0, -2],
                        [1, 0, -1]], dtype=float)
    sobel_y = np.array([[1, 2, 1],
                        [0, 0, 0],
                        [-1, -2, -1]], dtype=float)
    laplacian = np.array([[0, -1, 0],
                          [-1, 4, -1],
                          [0, -1, 0]], dtype=float)
    blur = np.ones((3, 3), dtype=float) / 9
    return [sobel_x, sobel_y, laplacian, blur]


def raw_vs_conv_accuracy(seed=0, kernels=None):
    if kernels is None:
        kernels = default_kernels()

    Xtr, Xte, ytr, yte = load_split(seed)
    raw_train = Xtr.reshape(len(Xtr), -1)
    raw_test = Xte.reshape(len(Xte), -1)
    conv_train = conv_features(Xtr, kernels)
    conv_test = conv_features(Xte, kernels)

    raw_model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(max_iter=5000))
    ])
    conv_model = Pipeline([
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression(max_iter=5000))
    ])

    raw_model.fit(raw_train, ytr)
    conv_model.fit(conv_train, ytr)

    raw_acc = raw_model.score(raw_test, yte)
    conv_acc = conv_model.score(conv_test, yte)

    return {
        "raw_acc": raw_acc,
        "conv_acc": conv_acc,
        "n_raw_features": raw_train.shape[1],
        "n_conv_features": conv_train.shape[1]
    }


In [3]:
img = np.arange(64.).reshape(8, 8)
print(im2col(img, 3).shape)
print(im2col(img, 3)[0])
ker = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1.]])
print(conv_via_im2col(img, ker).shape)
res = raw_vs_conv_accuracy(0)
print(res)


(36, 9)
[ 0.  1.  2.  8.  9. 10. 16. 17. 18.]
(6, 6)
{'raw_acc': 0.9688888888888889, 'conv_acc': 0.9755555555555555, 'n_raw_features': 64, 'n_conv_features': 36}


In [4]:
# Autograder-aligned checks
Xtr, Xte, ytr, yte = load_split(0)
assert Xtr.shape[1:] == (8, 8) and Xte.shape[1:] == (8, 8)
assert len(Xtr) + len(Xte) == 1797
assert im2col(img, 3).shape == (36, 9)
assert np.array_equal(im2col(img, 3)[0], np.array([0, 1, 2, 8, 9, 10, 16, 17, 18], dtype=float))

def direct_conv(image, kernel):
    H, W = image.shape
    kh, kw = kernel.shape
    out = np.zeros((H-kh+1, W-kw+1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(image[i:i+kh, j:j+kw] * kernel)
    return out

assert np.allclose(conv_via_im2col(img, ker), direct_conv(img, ker))
assert conv_features(Xtr[:10], default_kernels()[:2]).shape == (10, 18)
assert res["raw_acc"] > 0.85 and res["conv_acc"] > 0.85
assert res["n_raw_features"] == 64 and res["n_conv_features"] == 36
assert im2col(img, 1).shape == (64, 1)
assert np.array_equal(im2col(img, 1).reshape(-1), img.reshape(-1))
assert np.all(conv_features(Xtr[:10], default_kernels()) >= 0)
assert raw_vs_conv_accuracy(0) == res
print("All checks passed.")


All checks passed.
